# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all data elements by their unique `@id` as per the Croissant specification.

### Dataset Source
The dataset source is a Croissant schema, accessible via the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Make sure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print a basic overview
print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")
print(f"Cite as: {getattr(metadata, 'citeAs', None)}")

## 2. Data Overview
Review all **record sets** and their available **fields**, referencing each by their `@id` as per the Croissant schema.

In [ ]:
# List all record sets, fields, and columns with their `@id`

# Access metadata via mlcroissant's API; all referencing by '@id'
def print_recordsets_info(dataset):
    record_sets = dataset.metadata.record_sets
    print(f"Number of Record Sets: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print(f"  Name: {rs.name}")
        if hasattr(rs, 'description'):
            print(f"  Description: {rs.description}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id}")
            print(f"      Name: {field.name}")
            if hasattr(field, 'column'):
                columns = field.column if isinstance(field.column, list) else [field.column]
                for c in columns:
                    print(f"        Column @id: {c.id if hasattr(c, 'id') else str(c)}")
        print()
        print('-' * 60)

print_recordsets_info(dataset)

## 3. Data Extraction
Load data from each record set into Pandas DataFrames, using each record set's `@id` found above.
All extraction steps use the record set and field `@id` for precise referencing.

In [ ]:
# Extract data from all record sets into dataframes
dataframes = {}

# Collect all record set IDs
record_sets = dataset.metadata.record_sets
record_set_ids = [rs.id for rs in record_sets]
print(f"Available RecordSet @ids: {record_set_ids}")

for rs_id in record_set_ids:
    # Load all records for this record set
    records = list(dataset.records(record_set=rs_id))
    # Sometimes there may be no records (e.g. non-tabular or empty auxiliary sets)
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet {rs_id}, shape: {df.shape}")
        print("Available columns (field `@id`):", df.columns.tolist())
        print(df.head(2), "\n")
    else:
        print(f"RecordSet {rs_id} returned no records.")

# For demonstration, select the first non-empty RecordSet
main_record_set_id = None
for rs_id in record_set_ids:
    if rs_id in dataframes:
        main_record_set_id = rs_id
        break

if main_record_set_id:
    print(f"Proceeding with main RecordSet: {main_record_set_id}")
    main_df = dataframes[main_record_set_id]
    print(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping by field values, using only field and record set `@id`s for all references.

In [ ]:
# Identify a numeric field in the selected RecordSet
rs_obj = None
for rs in dataset.metadata.record_sets:
    if rs.id == main_record_set_id:
        rs_obj = rs
        break

# Find candidate numeric field(s) (by their @id)
numeric_field_id = None
for field in rs_obj.fields:
    # Choose a float or integer field (based on their declared type)
    if hasattr(field, 'data_type') and field.data_type in ['schema:Float', 'schema:Integer', 'Integer', 'Float', 'Number', 'schema:Number']:
        numeric_field_id = field.id
        print(f"Selected numeric field @id: {numeric_field_id} (name: {field.name})")
        break

if numeric_field_id is None:
    # Try to find any column containing only numbers
    import numpy as np
    for col in main_df.columns:
        if np.issubdtype(main_df[col].dtype, np.number):
            numeric_field_id = col
            print(f"Fallback: Using numeric column {col}")
            break

# Set thresholds for filtering
threshold = main_df[numeric_field_id].mean() if numeric_field_id else 0
print(f"Filtering {numeric_field_id} with threshold: {threshold}")

# Filter records by chosen numeric field
filtered_df = main_df
if numeric_field_id:
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize this numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a suitable categorical field (pick the first string/object-type field as example)
    group_field_id = None
    for field in rs_obj.fields:
        if field.id != numeric_field_id and hasattr(field, 'data_type') and field.data_type in ['schema:Text', 'Text', 'schema:String', 'String']:
            group_field_id = field.id
            break
    if not group_field_id:
        # Fallback: pick first object-type column
        for col in main_df.columns:
            if main_df[col].dtype == 'object' and col != numeric_field_id:
                group_field_id = col
                break

    if group_field_id and group_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and relationships between categorical and numeric fields, referencing only their `@id`.

In [ ]:
# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
if numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we have a group field, boxplot
    if 'group_field_id' in locals() and group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10, 4))
        # Only plot if the number of unique values is reasonable
        unique_groups = main_df[group_field_id].nunique()
        if unique_groups < 30:
            sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xticks(rotation=45)
            plt.show()
        else:
            print(f"Too many unique groups ({unique_groups}) to plot boxplot.")

## 6. Conclusion
This notebook demonstrated loading, inspecting, and analyzing the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all Croissant entities strictly by their `@id`. 

**Key takeaways:**
- The dataset includes multiple record sets with structured fields, allowing detailed protein abundance and modification studies.
- Numeric and categorical fields were processed and visualized entirely via their Croissant identifiers (`@id`).
- This standards-driven approach ensures reproducibility and enables programmatic exploration of FAIR datasets.

The same pattern can be extended for advanced analysis, direct ML, or pipeline integration, always referencing Croissant data entities by their `@id`.